теперь приступим к дообучению, устанавливаем библиотеку transformers для работы с моделями, импортируем автотокенезатор (у каждой модели свой токенизатор и просто так свой засунуть не получится,можно прописать его в ручуную,но если использовать его то токенизатор подберется автоматически к модели), берем модель (я выбрал руберта от МФТИ который обучен на русской википедии,он хорош по отзовам и немного весит,что позволит его быстро дообучить), затем для проверки можем использовать рандомный отзыв и увидеть как токенизатор его разбивает

In [31]:
%pip install transformers
%pip uninstall -y torch torchvision torchaudio
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

from transformers import AutoTokenizer

MODEL_NAME = 'DeepPavlov/rubert-base-cased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

sample_text = "Этот фильм вообще не понравился,полный отстой!"
tokens = tokenizer.tokenize(sample_text)

print(f"Оригинальный текст: {sample_text}")
print(f"Токены: {tokens}")


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Found existing installation: torch 2.10.0
Uninstalling torch-2.10.0:
  Successfully uninstalled torch-2.10.0
Found existing installation: torchvision 0.25.0
Uninstalling torchvision-0.25.0:
  Successfully uninstalled torchvision-0.25.0
Found existing installation: torchaudio 2.10.0
Uninstalling torchaudio-2.10.0:
  Successfully uninstalled torchaudio-2.10.0
Note: you may need to restart the kernel to use updated packages.


You can safely remove it manually.
You can safely remove it manually.
ERROR: Could not find a version that satisfies the requirement torch (from versions: none)

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for torch


Looking in indexes: https://download.pytorch.org/whl/cu121
Note: you may need to restart the kernel to use updated packages.
Оригинальный текст: Этот фильм вообще не понравился,полный отстой!
Токены: ['Этот', 'фильм', 'вообще', 'не', 'понравился', ',', 'полный', 'отстой', '!']


In [23]:
import pandas as pd
dt_train = pd.read_json("train.jsonl", lines=True)
dt_test = pd.read_json("test.jsonl", lines=True)
dt_val = pd.read_json("validation.jsonl", lines=True)
dt_full_train = pd.concat([dt_train, dt_val], ignore_index=True)

Можем посмотреть на encoding и decoding,видим,что все работает отлично (101 и 102 это обозначени для начала и конца предложения)

In [24]:
encoded_text = tokenizer.encode(sample_text)
print(f"encode: {encoded_text}")


decoded_text = tokenizer.decode(encoded_text)
print(f"decode: {decoded_text}")

encode: [101, 11514, 7142, 17050, 1699, 58171, 128, 21228, 110950, 106, 102]
decode: [CLS] Этот фильм вообще не понравился, полный отстой! [SEP]


Импортируем торч и достаем dataset и dataloader (первый просто шаблон для того чтобы торч мог читать данные, второй нужен для создания батчей), затем создаем класс ReviewDataset который наследуется от базового класса из pytorch, создаем метод init в котором мы передаем все наши данные (текст,лейблы,токенизатор и максималную длину текста), создаем функцию для определения количества отзовов (нужно для того что бы торч понимал на сколько бачей разбивать) и функцию для каждого нашего отзыва которая будет по сути циклом прогоняться каждый раз, мы обращаемся к таблице и вытаскиваем наш отзыв прописываем str на всякий случай если отзыв будет пустым, чтобы токенизатор не сломался и точно так же вытаскиваем лейбл этого отзыва, теперь же переходим к encoding самый важный момент, обращаемся к токенизатору и берем его оттуда и сохраняем его в нашу переменную,нам нужен сам текст,технические маячки(наподобии 101 и 102 для начала и конца текста), максимальная длина для того чтобы все матрицы были одного размера,отключим функцию return_token_type_ids которая нужна для задач подачи 2 текстов одновременно, затем padding и truncation которые будут подгонять под наш размер маленькие тексты и обрезать большие и поскольку padding добавляет кучу нулей мы сделаем маску вниманя через return_attention_mask чтобы берт смотрел только туда где есть 1,а 0 игнорировал ну и в конце нам нужно чтобы он отдал нам не просто список,а именно тензор торча. В конце нам нужно что бы нам выдало текст (для дальнейшей проверки), затем достаем наши закодированые слова (flatten нужен для того что бы он дал не матрицу,а строчку), достаем маску с таким же методом, в конце мы достаем наши лейблы которые превращаются в объект для pytorch и меняем их формат для удобной работы

In [25]:
import torch
from torch.utils.data import Dataset, DataLoader

class ReviewDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len 

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts.iloc[item])
        label = self.labels.iloc[item]

        encoding = self.tokenizer(
            text,
            add_special_tokens=True, 
            max_length=self.max_len, 
            return_token_type_ids=False,
            padding='max_length',   
            truncation=True,        
            return_attention_mask=True, 
            return_tensors='pt',     
        )

        return {
            'text': text,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

фиксируем длину каждого отзыва и размер батча, затем создаем переменную по нашему классу, передаем внутрь текст,лейблы,токенизатор и определяем лимит токенов, то же самое для теста,но на другой табличке, затем создаем loader для трейна и теста где передаем размер батчей (в трейне указываем перемешивание для борьбы с переобучением,в тесте нам это не нужно)

In [26]:
MAX_LEN = 128
BATCH_SIZE = 16


train_dataset = ReviewDataset(
    texts=dt_full_train['text'], 
    labels=dt_full_train['label'], 
    tokenizer=tokenizer, 
    max_len=MAX_LEN
)

test_dataset = ReviewDataset(
    texts=dt_test['text'], 
    labels=dt_test['label'], 
    tokenizer=tokenizer, 
    max_len=MAX_LEN
)


train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

импортируем берт для классификации, скачиваем нашу модель и даем ей слой для классификации на 3 выхода который мы и будем обучать, ищем девайс (видеокарту если не находит то цп), в моем случае он почему то не нашел видеокарту (возможно нужно установить другой pytorch),но данных мало и цп хватит, загружаем наши матрицы на девайс

In [27]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=3
)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


model.to(device)

print(f"отправил на: {device}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you

отправил на: cpu


Теперь берем оптимизатор, выберем AdamW(обычный Адам,но который борется с переобучением,что с таким количествам параметров для нас идеально), прописываем эпохи и лр (оба берем небольшие так как данных немного и 3 эпох будет достаточно,а лр берем максимально маленьким чтобы наша модель не подстроилась под наши отзывы), передаем все это в переменную вместе с параметрами и лр

In [28]:
from torch.optim import AdamW

EPOCHS = 3 
LEARNING_RATE = 0.00002

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)


In [30]:
from tqdm import tqdm # Библиотека для красивого прогресс-бара

# Запускаем цикл по количеству эпох
for epoch in range(EPOCHS):
    print(f"\nЭпоха {epoch + 1} / {EPOCHS}")
    
    # 1. Переводим модель в режим обучения
    model.train() 
    total_loss = 0
    
    # Обернем train_loader в tqdm, чтобы видеть полоску загрузки
    for batch in tqdm(train_loader):
        # 2. Достаем тензоры из батча и отправляем их на процессор/видеокарту
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        # 3. Обнуляем память оптимизатора (градиенты)
        optimizer.zero_grad()
        
        # 4. Прямой проход (Forward pass): просим модель сделать предсказание
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        
        # Удобная фишка BERT: если дать ему labels, он сам посчитает ошибку (Loss)
        loss = outputs.loss
        total_loss += loss.item()
        
        # 5. Обратный проход (Backward pass): вычисляем, где модель ошиблась
        loss.backward()
        
        # 6. Шаг оптимизатора: AdamW подкручивает веса
        optimizer.step()
        
    # Считаем среднюю ошибку за всю эпоху
    avg_loss = total_loss / len(train_loader)
    print(f"Средняя ошибка (Loss): {avg_loss:.4f}")


Эпоха 1 / 3


  0%|          | 0/750 [00:03<?, ?it/s]


KeyboardInterrupt: 